# Planificateur de menus a l'echelle reelle — theoreme hierarchique `int[][]`

> Capstone de la serie SMT / Z3.Linq — volet **structure hierarchique** (voir #4617).

Le notebook [05 — Meal Planner Hierarchical](05_Meal_Planner_Hierarchical.ipynb) demontre le **theoreme hierarchique** de Z3.Linq sur un corpus *jouet* : un plan de repas modelise comme un tableau en escalier `int[][]` (jours x creneaux), avec bornes de categorie par creneau, contrainte de montee en gamme, et permutation totale (`Distinct` cross-row en mode `CollectionHandling.Array`).

Ce notebook **05b** reprend exactement ce theoreme, mais le nourrit avec un **corpus reel** : la base de composition nutritionnelle **Ciqual (ANSES 2025)** fusionnee aux **menus de cantine** (corpus *Recettes*, qui porte la structure en creneaux `ordre_plat`). On passe de quelques plats jouets a **138 plats reels** repartis sur 7 creneaux.

Il complete le notebook [07b — Real Corpus Scaling](07b_Meal_Planner_Real_Corpus_Scaling.ipynb), qui explore l'axe orthogonal : l'**encodage nutritionnel a l'echelle** (one-hot pseudo-booleen) sur le corpus RecipeML. Ici, l'enjeu n'est pas la nutrition mais la **structure combinatoire hierarchique** : selectionner, parmi un vrai catalogue, un plan de 7 jours qui respecte le role de chaque creneau et la diversite, le tout en **LINQ pur sur un `int[][]`**.

**Plan**
1. Le corpus reel et sa structure en creneaux
2. Le theoreme hierarchique `int[][]` — deux variantes (montee en gamme / permutation)
3. Exercices

## Prerequis et provenance des donnees

Le corpus est **trop volumineux pour etre versionne** : il se telecharge a la demande dans `data/meals/` (gitignore) avec le script [`download_meal_data.py`](download_meal_data.py) :

```bash
python download_meal_data.py --ciqual --recettes
```

La fusion **Ciqual x Recettes** (jointure lexicale sac-de-mots denree -> aliment Ciqual, puis aggregation des compositions par plat) est realisee par `Dietetique.Load` (port du projet *Demo2* de `Z3.LinqBinding`). Elle produit un cache **`dietetique.json`** que l'on charge ici directement, afin de concentrer ce notebook sur le **theoreme hierarchique**. Le detail de la jointure lexicale Ciqual (lecture en flux du fichier de composition de 69 Mo, index inverse mot -> aliment) est documente pas-a-pas dans le notebook [07b](07b_Meal_Planner_Real_Corpus_Scaling.ipynb).

> **Doctrine « decider ou echouer bruyamment »** : si le cache est absent, la cellule de chargement l'**annonce explicitement** et indique la commande a lancer — elle ne fabrique jamais un corpus de substitution.

In [1]:
#r "../Z3.Linq/.deploy/Microsoft.Z3.dll"
#r "../Z3.Linq/.deploy/ExpressionUtils.dll"
#r "../Z3.Linq/.deploy/Z3.Linq.dll"
using Z3.Linq;
using Microsoft.Z3;
using System;
using System.IO;
using System.Linq;
using System.Collections.Generic;
using System.Text.Json;
using System.Diagnostics;

// POCOs miroir du schema dietetique.json (couche de fusion Ciqual x Recettes).
public class Dietetique
{
    public List<Constituant> Constituants { get; set; }
    public List<DenreeImport> Denrees { get; set; }
    public List<PlatImport> Plats { get; set; }
}
public class Constituant { public string Nom { get; set; } }
public class DenreeImport { public string Nom { get; set; } = ""; public decimal[] Compositions { get; set; } }
public class Ingredient { public int Denree { get; set; } public decimal Quantite { get; set; } }
public class PlatImport
{
    public string Nom { get; set; }
    public int Ordre { get; set; }
    public List<Ingredient> Ingredients { get; set; } = new List<Ingredient>();
    public decimal[] Compositions { get; set; }
}

var DIET = "data/meals/dietetique.json";
Dietetique diet = null;
if (!File.Exists(DIET))
{
    Console.WriteLine("Cache 'dietetique.json' absent — echec bruyant (pas de corpus de substitution).");
    Console.WriteLine("Lancez d'abord : python download_meal_data.py --ciqual --recettes");
    Console.WriteLine("La fusion Ciqual x Recettes (Dietetique.Load) produira ensuite le cache.");
}
else
{
    diet = JsonSerializer.Deserialize<Dietetique>(File.ReadAllText(DIET));
    Console.WriteLine($"Corpus charge : {diet.Constituants.Count} constituants, "
        + $"{diet.Denrees.Count} denrees appariees, {diet.Plats.Count} plats.");
}

The below script needs to be able to find the current output cell; this is an easy method to get it.

Corpus charge : 74 constituants, 198 denrees appariees, 138 plats.


## 1. Le corpus reel et sa structure en creneaux

Chaque plat porte un **creneau** (`Ordre`, de 1 a 7) : entree, plat principal, accompagnement, laitage/fromage, dessert, etc. C'est cette structure qui rend le probleme *hierarchique* — un plan de repas n'est pas un sac de plats interchangeables, chaque jour doit servir **un plat par creneau, du bon type**.

Pour rendre chaque creneau adressable par une **plage d'index contigue** (exactement comme les bornes de categorie `[entLo, entHi]` du notebook 05 jouet), on **trie les plats par creneau**. On se restreint aux creneaux 1 a 5 (les plus garnis), suffisants pour un plan equilibre de 7 jours.

In [2]:
const int Jours = 7;
const int Creneaux = 5;   // ordres 1..5

// Trie par creneau -> chaque ordre occupe une plage d'index contigue [lo[o]..hi[o]].
var plats = diet.Plats.Where(p => p.Ordre >= 1 && p.Ordre <= Creneaux)
                      .OrderBy(p => p.Ordre).ThenBy(p => p.Nom).ToList();

var lo = new int[Creneaux + 1];
var hi = new int[Creneaux + 1];
string[] roles = { "", "entree", "plat principal", "accompagnement", "laitage/fromage", "dessert" };
for (int o = 1; o <= Creneaux; o++)
{
    lo[o] = plats.FindIndex(p => p.Ordre == o);
    hi[o] = plats.FindLastIndex(p => p.Ordre == o);
    Console.WriteLine($"  creneau {o} ({roles[o],-16}) : index [{lo[o]}..{hi[o]}]  ({hi[o] - lo[o] + 1} plats)");
}
Console.WriteLine($"\nTotal exploitable : {plats.Count} plats sur {Creneaux} creneaux.");
Console.WriteLine("\nExemples (creneau 1) : " + string.Join(", ",
    plats.Where(p => p.Ordre == 1).Take(4).Select(p => p.Nom.Trim())));

  creneau 1 (entree          ) : index [0..28]  (29 plats)


  creneau 2 (plat principal  ) : index [29..65]  (37 plats)


  creneau 3 (accompagnement  ) : index [66..86]  (21 plats)


  creneau 4 (laitage/fromage ) : index [87..107]  (21 plats)


  creneau 5 (dessert         ) : index [108..121]  (14 plats)



Total exploitable : 122 plats sur 5 creneaux.



Exemples (creneau 1) : CAROTTES A L'ORANGE, CAROTTES RAPEES(S/V), CELERI REMOULADE, GOUGERE INDIVIDUELLE


## 2. Le theoreme hierarchique `int[][]`

On modelise le plan hebdomadaire comme un **tableau en escalier** `int[][]` : `Plan[j][c]` = l'index (dans la liste triee `plats`) du plat servi le jour `j` au creneau `c`. C'est la **meme structure** que le notebook 05 jouet, mais a l'echelle reelle.

Les helpers sont definis dans une **classe statique** (`Menu`) : dans un noyau .NET Interactive, une classe statique **persiste d'une cellule a l'autre**, contrairement a une fonction locale. Le theoreme et les exercices la reutilisent.

- `WithBounds` impose, pour chaque jour et chaque creneau, que l'index reste dans la plage du bon creneau (`Plan[j][c] in [lo[c+1], hi[c+1]]`). C'est la contrainte de categorie, exprimee une fois pour les `7 x 5 = 35` cases.
- `PrintPlan` affiche le plan resolu en remontant des index aux noms de plats.

In [3]:
// Plan hebdomadaire : 7 jours x 5 creneaux, modelise comme int[][] (jagged).
public class WeeklyMenuPlan
{
    public int[][] Plan { get; set; } = new int[7][];
    public WeeklyMenuPlan() { for (int j = 0; j < 7; j++) Plan[j] = new int[5]; }
}

// Classe statique -> persiste entre cellules. Reutilisee par le theoreme ET les exercices.
public static class Menu
{
    public const int Jours = 7;
    public const int Creneaux = 5;

    // Bornes de categorie : Plan[j][c] doit indexer un plat du creneau (c+1).
    public static Theorem<WeeklyMenuPlan> WithBounds(Z3Context ctx, int[] lo, int[] hi)
    {
        var t = ctx.NewTheorem<WeeklyMenuPlan>();
        for (int j = 0; j < Jours; j++)
        {
            int jj = j;
            for (int c = 0; c < Creneaux; c++)
            {
                int cc = c, o = c + 1;
                int basLo = lo[o], basHi = hi[o];   // capture en constantes pour la traduction Z3
                t = t.Where(p => p.Plan[jj][cc] >= basLo && p.Plan[jj][cc] <= basHi);
            }
        }
        return t;
    }

    public static void PrintPlan(WeeklyMenuPlan plan, List<PlatImport> plats)
    {
        for (int j = 0; j < Jours; j++)
        {
            var noms = Enumerable.Range(0, Creneaux).Select(c => plats[plan.Plan[j][c]].Nom.Trim());
            Console.WriteLine($"  Jour {j + 1} : " + string.Join(" | ",
                noms.Select(n => n.Length > 22 ? n.Substring(0, 22) : n)));
        }
    }
}
Console.WriteLine("Helpers definis : WeeklyMenuPlan, Menu.WithBounds, Menu.PrintPlan.");

Helpers definis : WeeklyMenuPlan, Menu.WithBounds, Menu.PrintPlan.


In [4]:
// --- Variante A : montee en gamme (mode Array implicite) ---
// Pour chaque creneau, l'index du plat croit strictement au fil de la semaine :
// une selection ordonnee, sans repetition, qui "monte en gamme" jour apres jour.
using (var ctx = new Z3Context())
{
    var th = Menu.WithBounds(ctx, lo, hi);
    for (int c = 0; c < Creneaux; c++)
    {
        int cc = c;
        for (int j = 0; j < Jours - 1; j++)
        {
            int jj = j;
            th = th.Where(p => p.Plan[jj][cc] < p.Plan[jj + 1][cc]);
        }
    }
    var sw = Stopwatch.StartNew();
    var plan = th.Solve();
    sw.Stop();
    Console.WriteLine($"[A] Montee en gamme resolue en {sw.Elapsed.TotalMilliseconds:F0} ms :\n");
    if (plan != null) Menu.PrintPlan(plan, plats); else Console.WriteLine("  UNSAT");
}

[A] Montee en gamme resolue en 85 ms :



  Jour 1 : RADIS BEURRE | BOULETTES SOJA | CAROTTE RONDELLES (BTE | BLEU FRANCAIS 25G | ANANAS SIROP MORCEAUX


  Jour 2 : RILLETTES DE LA MER IN | BOULETTES SOJA SCE TOM | CHOU FLEUR BECHAMEL | BREBICREME 20G | BISCUIT FOURRE CHOCOLA


  Jour 3 : SAL COLESLAW CC | BRANDADE | CHOU FLEUR PERSILLE | CABECOU BIOLOGIQUE | BOUDOIR


  Jour 4 : SAL D  HIVER (crudités | BROCHETTE DE DINDONNEA | COQUILLETTES BEURRE | CAMEMBERT PORTION  30G | COMPOTE IND POMME SUD 


  Jour 5 : SAL ENDIVES | CUBES DE COLIN CREME C | COURGETTES BECHAMEL | CANTAL PORTION 25G | COMPOTE IND POMME/ABRI


  Jour 6 : SAL MELANGE TENDRE | CUBES POIS POIVRONS | EPINARDS BECHAMEL | CHANTENEIGE | COMPOTE IND POMME/ANAN


  Jour 7 : SAL P DE TERRE (PARMEN | CUBES SAUMON A L ANETH | FLAGEOLETS PERSILLES | CHEVRE PETIT CABRAY | COMPOTE IND POMME/MIRA


### Interpretation — variante A

La contrainte de montee en gamme (`Plan[j][c] < Plan[j+1][c]`) force, pour chaque creneau, une **suite d'index strictement croissante** sur les 7 jours. Comme les plats sont tries par creneau, cela revient a parcourir le catalogue dans l'ordre, sans jamais reservir le meme plat. Z3.Linq traduit le `int[][]` en variables entieres et resout en LINQ pur, sans expansion manuelle des paires.

In [5]:
// --- Variante B : permutation totale (CollectionHandling.Array EXPLICITE, Distinct cross-row) ---
// Chaque creneau est servi 7 fois dans la semaine, sans aucune repetition :
// Z3Methods.Distinct s'applique directement aux acces cross-row d'un int[][]
// (meme marqueur que les colonnes d'un Sudoku) -> une contrainte globale SMT.
using (var ctxPerm = new Z3Context { DefaultCollectionHandling = CollectionHandling.Array })
{
    var perm = Menu.WithBounds(ctxPerm, lo, hi);
    for (int c = 0; c < Creneaux; c++)
    {
        int cc = c;
        perm = perm.Where(p => Z3Methods.Distinct(
            p.Plan[0][cc], p.Plan[1][cc], p.Plan[2][cc], p.Plan[3][cc],
            p.Plan[4][cc], p.Plan[5][cc], p.Plan[6][cc]));
    }
    var sw = Stopwatch.StartNew();
    var planPerm = perm.Solve();
    sw.Stop();
    Console.WriteLine($"[B] Permutation totale (Distinct cross-row) resolue en {sw.Elapsed.TotalMilliseconds:F0} ms :\n");
    if (planPerm != null) Menu.PrintPlan(planPerm, plats); else Console.WriteLine("  UNSAT");
}

[B] Permutation totale (Distinct cross-row) resolue en 31 ms :



  Jour 1 : RADIS BEURRE | BOULETTES SOJA | CAROTTE RONDELLES (BTE | BLEU FRANCAIS 25G | ANANAS SIROP MORCEAUX


  Jour 2 : RILLETTES DE LA MER IN | BOULETTES SOJA SCE TOM | CHOU FLEUR BECHAMEL | BREBICREME 20G | BISCUIT FOURRE CHOCOLA


  Jour 3 : SAL COLESLAW CC | BRANDADE | CHOU FLEUR PERSILLE | CABECOU BIOLOGIQUE | BOUDOIR


  Jour 4 : SAL D  HIVER (crudités | BROCHETTE DE DINDONNEA | COQUILLETTES BEURRE | CAMEMBERT PORTION  30G | COMPOTE IND POMME SUD 


  Jour 5 : SAL ENDIVES | CUBES SAUMON A L ANETH | COURGETTES BECHAMEL | CANTAL PORTION 25G | COMPOTE IND POMME/ABRI


  Jour 6 : SAL MELANGE TENDRE | CUBES DE COLIN CREME C | EPINARDS BECHAMEL | CHANTENEIGE | COMPOTE IND POMME/ANAN


  Jour 7 : SAL P DE TERRE (PARMEN | CUBES POIS POIVRONS | FLAGEOLETS PERSILLES | CHEVRE PETIT CABRAY | COMPOTE IND POMME/MIRA


### Interpretation — variante B

La variante B n'impose plus d'ordre, seulement la **diversite** : chaque creneau est servi sans repetition sur la semaine. `Z3Methods.Distinct` appliquee aux acces *cross-row* d'un `int[][]` produit **une seule contrainte SMT globale** par creneau (et non `C(7,2)` inegalites deux-a-deux) — exactement le mecanisme qui modelise les colonnes d'un Sudoku.

Le mode `CollectionHandling.Array` est ici **explicite** : il indique a Z3.Linq de reifier le tableau comme un tableau SMT, ce qui rend les acces cross-row adressables par `Distinct`. C'est le levier qui fait passer le `int[][]` du statut de simple structure de donnees a celui de **variable de decision hierarchique**.

## 3. Exercices

Les trois exercices ci-dessous se completent en ajoutant des contraintes au theoreme. Ils reutilisent les helpers `Menu.WithBounds` / `Menu.PrintPlan` et les tableaux `lo` / `hi` / `plats` deja en place.

In [6]:
// Exercice 1 — Diversite renforcee : pas deux fois la meme entree (creneau 1)
//              sur deux jours CONSECUTIFS.
//
// Indice : ajoutez, pour j de 0 a Jours-2, une contrainte
//          p.Plan[j][0] != p.Plan[j+1][0]  sur un theoreme construit avec Menu.WithBounds.
// Etape 1 : creez un Z3Context (mode Array implicite suffit).
// Etape 2 : partez de Menu.WithBounds(ctx, lo, hi), ajoutez la contrainte d'inegalite consecutive.
// Etape 3 : resolvez et affichez avec Menu.PrintPlan.
//
// TODO etudiant : completez ci-dessous.
Console.WriteLine("Exercice 1 a completer : diversite des entrees sur jours consecutifs.");

Exercice 1 a completer : diversite des entrees sur jours consecutifs.


In [7]:
// Exercice 2 — Bande nutritionnelle (avance).
// Chaque plat porte un vecteur Compositions[] (74 constituants, aligne sur diet.Constituants).
// Objectif : contraindre l'energie totale d'une journee dans une bande [min, max].
//
// Difficulte : l'index Plan[j][c] est SYMBOLIQUE -> on ne peut pas ecrire plats[Plan[j][c]].Compositions[k]
//              directement. C'est exactement le probleme resolu dans le notebook 07b par un
//              encodage one-hot pseudo-booleen (une variable binaire par (creneau, plat)).
// Indice : reportez-vous a 07b pour l'encodage one-hot, puis sommez l'energie selectionnee par jour.
// Etape 1 : reperez l'index du constituant "Energie" dans diet.Constituants.
// Etape 2 : pour chaque jour, exprimez la somme d'energie des 5 plats choisis.
// Etape 3 : contraignez cette somme dans une bande realiste, resolvez.
//
// TODO etudiant : completez ci-dessous (encodage one-hot inspire de 07b).
Console.WriteLine("Exercice 2 a completer : bande nutritionnelle via encodage one-hot (cf 07b).");

Exercice 2 a completer : bande nutritionnelle via encodage one-hot (cf 07b).


In [8]:
// Exercice 3 — Restrictions patient (Min/Max par constituant) sur un creneau supplementaire.
// Le corpus contient un 6e et un 7e creneau (Ordre 6 et 7) actuellement ignores.
//
// Indice : passez Creneaux a 6 (ou 7), regenerez lo/hi sur la nouvelle plage, et adaptez
//          WeeklyMenuPlan pour 6 (ou 7) colonnes. Verifiez que chaque creneau a assez de plats
//          pour 7 jours distincts (variante B) — sinon le probleme devient UNSAT (echec honnete).
// Etape 1 : recomptez la distribution des plats pour Ordre 6 et 7.
// Etape 2 : decidez quels creneaux ajouter sans rendre la permutation infaisable.
// Etape 3 : re-resolvez et commentez la faisabilite.
//
// TODO etudiant : completez ci-dessous.
Console.WriteLine("Exercice 3 a completer : creneau supplementaire et faisabilite de la permutation.");

Exercice 3 a completer : creneau supplementaire et faisabilite de la permutation.


## Conclusion

Ce notebook a montre que le **theoreme hierarchique `int[][]`** de Z3.Linq — bornes de categorie par creneau, montee en gamme, permutation totale par `Distinct` cross-row en mode `CollectionHandling.Array` — passe **sans modification** du corpus jouet (notebook 05) a un **corpus reel** de 138 plats issus de la fusion Ciqual 2025 x Recettes.

L'interet pedagogique est double :

- **La structure prime sur l'echelle.** Le meme `int[][]` modelise un plan de 7 jours sur un vrai catalogue ; c'est la *structure en creneaux* (et non le nombre de plats) qui fait du probleme un probleme hierarchique. Z3 resout les deux variantes en quelques dizaines de millisecondes.
- **Separation des preoccupations.** Ce notebook traite la **combinatoire hierarchique** ; le notebook [07b](07b_Meal_Planner_Real_Corpus_Scaling.ipynb) traite l'**encodage nutritionnel a l'echelle** (one-hot pseudo-booleen). L'exercice 2 fait le pont entre les deux.

**Pour aller plus loin** : combiner la structure hierarchique (ce notebook) avec les contraintes nutritionnelles (07b) donne un planificateur de menus complet — selection structuree *et* equilibree — entierement exprime en LINQ sur des `int[][]`, resolu par Z3.

---

*Serie SMT / Z3.Linq — voir le [README](README.md) pour le parcours complet.*